# Spark-X2.6-4B-En4B-A1B (`v0.3`) — Diagnostic Probing & Roadmap Gate Testing Notebook

This dedicated probing notebook lets you interactively test and inspect every gate in [`docs/roadmap/README.md`](docs/roadmap/README.md) on top of the **72L Loop Base (`LB-blend`)**:
- **Cell 1**: Clone / Sync Repo, Install Dependencies & Run Unit Test Suite (`tests/test_memory_math.py`, Gates `G0`–`G5`).
- **Cell 2**: Load `TassP0lar/Spark-X2.6-4B-En4B-A1B`, Apply Cooperative Blend (`LB-blend`: $\beta_{\text{interface}}=0.30, \beta_{\text{body}}=0.12$), & Capture `BackboneFingerprint` (`Gate G0`).
- **Cell 3**: **Track L Probe (`Gates GL1–GL5`)** — Diagnose `TwoStreamMHC` P2/P3 gains & pass-wise KL logit lens, repair harmful Engram anchors (`GL2`), enforce softplus gain floor (`GL3`), measure adaptive loop halting (`GL4`), and verify recursion-wise global KV sharing (`GL5`).
- **Cell 4**: **Phase 1 & Phase 2 Probe (`Gates G1 & G2`)** — Verify exact Spark gated attention output reconstruction (`error < 1e-3`), calibrate RoPE-free `[64:256]` `D-LSA` index (`fit_dlsa_index`), measure attention-mass recall@K, and test `ColdKVStore` Ward bucketization & page fetch.
- **Cell 5**: **Phase 3, 4, 5, 6 & 7 Probe (`Gates G3–G6, EFRE, H-L1/H-L2`)** — Run `run_efre_and_loop_manipulation_probe` (`ESM` + `LRCD` compilation, `72L Loop Base vs 36L loop-off` manipulation comparison, and bit-exact `SleepScheduler.forget()` rollback) + `consolidate_shards_smf` (`G6`), printing a single compact JSON summary block you can paste directly back into chat.


In [ ]:
# ==============================================================================
# Cell 1: Clone / Sync Repository, Install Dependencies & Run Unit Test Suite
# ==============================================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/BaiTian6641/Spark-X2.6-Recursion.git"
REPO_DIR = Path("/content/Spark-X2.6-Recursion")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.46.0", "accelerate>=0.34.0", "datasets>=3.0.0",
    "huggingface_hub>=0.26.0", "safetensors>=0.4.5", "sentencepiece", "sympy"
], check=True)

print("Running Phase 0-5 Unit Test Suite (`tests/test_memory_math.py`)...")
res = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    capture_output=True, text=True,
)
print(res.stdout)
if res.stderr:
    print(res.stderr)
if res.returncode != 0:
    raise RuntimeError(f"Unit tests failed with exit code {res.returncode}")


In [ ]:
# ==============================================================================
# Cell 2: Load `TassP0lar/Spark-X2.6-4B-En4B-A1B`, Apply Cooperative Blend (`LB-blend`)
#         & Capture Gate G0 `BackboneFingerprint`
# ==============================================================================
import importlib, json, os, subprocess, torch
subprocess.run(["git", "pull", "--ff-only"], check=False)
import sparkx_memory_utils
import sparkx_loop_v03_core
import sparkx_loop_v03_benchmarks as bench
import load_inference_v03
importlib.reload(sparkx_memory_utils)
importlib.reload(sparkx_loop_v03_core)
importlib.reload(bench)
importlib.reload(load_inference_v03)

from load_inference_v03 import load_sparkx_loop_v03
from sparkx_loop_v03_core import SparkXLoopV03

def _get_secret(name: str):
    val = os.environ.get(name)
    if val:
        return val.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    return None

HF_REPO_ID = "TassP0lar/Spark-X2.6-4B-En4B-A1B"
BASE_MODEL_ID = "XHToken/Spark-X2.6-4B"
HF_TOKEN = _get_secret("HF_TOKEN")

if "model" not in globals() or model is None:
    model, tokenizer = load_sparkx_loop_v03(
        repo_id=HF_REPO_ID,
        device="cuda" if torch.cuda.is_available() else "cpu",
        token=HF_TOKEN,
        enable_loglinear_kv=True,
        use_pristine_base=False,
    )
else:
    print(f"[VRAM Cache Hit] Reusing already-loaded `{HF_REPO_ID}` model in GPU memory.")

# Hot-bind latest methods onto live model instance
for _m_name in (
    "_layers", "loop_config", "set_mhc_gain_floor", "get_mhc_controller_telemetry",
    "set_recursion_kv_sharing", "capture_residuals", "capture_attention_io",
    "reset_sleep_time_memory_compiler", "run_sleep_time_memory_compiler",
):
    if hasattr(SparkXLoopV03, _m_name):
        setattr(type(model), _m_name, getattr(SparkXLoopV03, _m_name))

# Apply Cooperative Task-Vector Blend (`LB-blend`: beta_interface=0.30, beta_body=0.12)
blend_report = bench.restore_pristine_base_backbone(
    model=model,
    base_model_id=BASE_MODEL_ID,
    blend_interface_alpha=0.30,
    blend_body_alpha=0.12,
    soft_reset_mhc_gain=0.35,
    token=HF_TOKEN,
)
fp_g0 = sparkx_memory_utils.BackboneFingerprint(model, include_controllers=False, store_exact_clone=False)
print("[Gate G0 Fingerprint Captured]", json.dumps(blend_report, indent=2))


In [ ]:
# ==============================================================================
# Cell 3: Track L Probe (`Gates GL1-GL5`) — Loop & Engram Health, Anchor Repair,
#         Softplus Gain Floor, Adaptive Loop Halting & Recursion KV Sharing
# ==============================================================================
import json
import sparkx_loop_v03_benchmarks as bench

gl3_floor = model.set_mhc_gain_floor(min_gain_p2=0.05, min_gain_p3=0.02)
gl5_kv_share = model.set_recursion_kv_sharing(enabled=True)
gl2_repair = bench.repair_harmful_engram_anchors(model, tokenizer)
gl4_exit = bench.evaluate_adaptive_loop_exit(model, tokenizer, kl_threshold=0.015)
model.set_recursion_kv_sharing(enabled=False)

track_l_summary = {
    "GL2_anchor_repair": {
        "no_engram_72L_ppl": gl2_repair["no_engram_72L_ppl"],
        "repaired_engram_72L_ppl": gl2_repair["repaired_engram_72L_ppl"],
        "zeroed_anchors": gl2_repair["zeroed_anchors"],
        "gl2_passed": gl2_repair["gl2_passed"],
        "logit_lens_by_pass": gl2_repair["initial_diagnostics"]["logit_lens_by_pass"],
    },
    "GL3_mhc_gain_floor": gl3_floor,
    "GL4_adaptive_loop_exit": gl4_exit,
    "GL5_recursion_kv_sharing": gl5_kv_share,
}
print("=== TRACK L PROBE SUMMARY ===")
print(json.dumps(track_l_summary, indent=2))


In [ ]:
# ==============================================================================
# Cell 4: Phase 1 & Phase 2 Probe (`Gates G1 & G2`) — Attention Output Reconstruction,
#         RoPE-Free `[64:256]` D-LSA Index Calibration & ColdKVStore Ward Paging
# ==============================================================================
import json
import sparkx_memory_utils as mem

device = model._head_weight().device
sample_text = (
    "Spark-X2.6-4B hybrid attention combines 27 sliding-window RoPE layers with 9 global "
    "partial-RoPE layers whose unrotated 192-D slice [64:256] provides position-free D-LSA indexing."
)
sample_ids = tokenizer(sample_text, return_tensors="pt")["input_ids"].to(device)

# Gate G1: Attention output reconstruction (< 1e-3 relative L2 error)
attn_io = model.capture_attention_io(sample_ids, layers=[3, 7, 15, 31], visit=0)
g1_errors = {f"L{k}": round(float(v["reconstruction_rel_error"]), 6) for k, v in attn_io.items()}
g1_passed = all(float(v["reconstruction_rel_error"]) < 1e-3 for v in attn_io.values())

# Gate G2: D-LSA Index Calibration & Recall@K + ColdKVStore
dlsa_fit = mem.fit_dlsa_index(model, [sample_ids], layers=[3, 7, 15, 31], steps=15, k_budget=32)
cold_store = mem.ColdKVStore(page_size=64, d_idx=64, max_bucket=4)
io_l3 = attn_io[3]
lsa_l3 = model._layers()[3].self_attn.lsa_indexer
k_s63 = lsa_l3.project_keys(io_l3["key"][0].permute(1, 0, 2).contiguous())
q_s63 = lsa_l3.project_queries(io_l3["query"][0].permute(1, 0, 2).contiguous())
page_id = cold_store.append_page(io_l3["key"][0], io_l3["value"][0], k_s63)
cold_fetch = cold_store.fetch_selected_kv(q_s63[-1:], top_buckets=2)

g1_g2_summary = {
    "G1_attention_reconstruction_errors": g1_errors,
    "G1_passed": g1_passed,
    "G2_dlsa_fit": dlsa_fit,
    "G2_cold_kv_store": {
        "page_id": page_id,
        "selected_tokens": cold_fetch["selected_tokens"],
        "bytes_transferred": cold_fetch["bytes_transferred"],
    },
}
print("=== PHASE 1 & PHASE 2 PROBE SUMMARY ===")
print(json.dumps(g1_g2_summary, indent=2))


In [ ]:
# ==============================================================================
# Cell 5: Phase 3-7 End-to-End Probe (`Gates G3, G4, G5, G6, EFRE & H-L1/H-L2`)
#         + Unified Copy-Paste Telemetry JSON
# ==============================================================================
import json, torch
import sparkx_memory_utils as mem
import sparkx_loop_v03_benchmarks as bench

efre_report = bench.run_efre_and_loop_manipulation_probe(model, tokenizer, eta=0.75)

# Gate G6: Deep-Sleep SMF Consolidation from ESM Shard -> MoE-Engram
device = model._head_weight().device
esm_l3 = model.esm["3"]
h_syn = torch.randn(4, esm_l3.width, device=device, dtype=torch.float32) * 0.1
phi_syn = esm_l3.features(h_syn, shard=0)
esm_l3.write(shard=0, A=phi_syn, Delta=h_syn * 0.05, alpha=1e-2, centroids=esm_l3.route_embed(h_syn[:1]))
smf_rep = mem.consolidate_shards_smf(model, shard_ids=(0,), remove_consolidated_shards=True)

# Verify Gate G0 Backbone Invariance across all probes
g0_verify = fp_g0.verify(model)

full_probe_telemetry = {
    "G0_backbone_verification": g0_verify,
    "Track_L": globals().get("track_l_summary", {}),
    "Phase_1_2": globals().get("g1_g2_summary", {}),
    "Phase_3_4_5_7_EFRE": efre_report,
    "Phase_6_SMF": {
        "anchors_consolidated": smf_rep.anchors_consolidated,
        "shards_consolidated": smf_rep.shards_consolidated,
        "mean_rel_fit_error": round(smf_rep.mean_rel_fit_error, 6),
        "max_null_space_leakage": smf_rep.max_null_space_leakage,
        "backbone_sha256_match": smf_rep.backbone_sha256_match,
    },
}
print("\n" + "=" * 90)
print("=== COPY-PASTE PROBE TELEMETRY JSON ===")
print(json.dumps(full_probe_telemetry, indent=2))
print("=" * 90)
